In [ ]:
!apt-get update -qq
!apt-get install -y -qq nginx

!pip install -q virtualenv
# Cloudflare tunnel
!curl -L -o cloudflared.deb \
    https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb

!dpkg -i cloudflared.deb

print("✅ Installation complete!")

In [ ]:
!python -m virtualenv /kaggle/working/venv

In [ ]:
!/kaggle/working/venv/bin/python --version

In [ ]:
!/kaggle/working/venv/bin/python -m pip install wrapt
!/kaggle/working/venv/bin/python -m pip install -U pip setuptools wheel

In [ ]:
!/kaggle/working/venv/bin/python -m pip install -q \
    vllm \
    fastapi \
    uvicorn \
    "uvicorn[standard]"

In [ ]:
import os
import subprocess
import time
import urllib.request

os.environ["VLLM_ENGINE_ITERATION_TIMEOUT_S"] = "300"

VENV_PATH = "/kaggle/working/venv"
PYTHON_EXECUTABLE = os.path.join(VENV_PATH, "bin", "python")
VLLM_EXECUTABLE = os.path.join(VENV_PATH, "bin", "vllm")

# -------------------------------------------------------------
# Configuration
# -------------------------------------------------------------

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct-AWQ"
MODEL_PORT = 8001


if "chat_process" not in globals():
    chat_process = None

    
# -------------------------------------------------------------
# Stop previous process and clean Ports
# -------------------------------------------------------------

def stop_servers(llm_process, process_name, process_port):
    if process_name in globals() and llm_process is not None:
        if llm_process.poll() is None:
            print(f"🛑 Stopping previous {process_name} server (PID: {llm_process.pid})...")
            llm_process.terminate()
            try:
                llm_process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                llm_process.kill()
                llm_process.wait()

    result = subprocess.run(["lsof", "-t", f"-i:{process_port}"], capture_output=True, text=True)
    if result.stdout.strip():
        pids = result.stdout.strip().splitlines()
        for pid in pids:
            subprocess.run(["kill", "-9", pid])
        time.sleep(1)
    print(f"✅ Port {process_port} is free.")

stop_servers(chat_process, "chat_process", MODEL_PORT)

# -------------------------------------------------------------
# Health Check Wait Function
# -------------------------------------------------------------
def wait_for_server(port, timeout=180, sleep=4):
    url = f"http://127.0.0.1:{port}/health"
    start_time = time.time()
    print(f"⏳ Waiting for model on port {port} to pass health checks...", end="")
    while time.time() - start_time < timeout:
        try:
            with urllib.request.urlopen(url, timeout=2) as response:
                if response.status == 200:
                    print(" Ready! ✅")
                    return True
        except Exception:
            pass
        print(".", end="", flush=True)
        time.sleep(sleep)
    print(" TIMED OUT OR CRASHED! ❌")
    return False

print("\n🚀 Starting vLLM Server...")
print(f"\n Launching Chat Model on Port {MODEL_PORT} (GPU 0)...")
env_gpu0 = os.environ.copy()
env_gpu0["CUDA_VISIBLE_DEVICES"] = "0"
chat_process = subprocess.Popen([
    VLLM_EXECUTABLE, 
    "serve", MODEL_NAME,
    "--quantization", "awq",
    "--port", str(MODEL_PORT),
    "--gpu-memory-utilization", "0.90",
    "--max-model-len", "4096",
    "--enforce-eager"
], env=env_gpu0, stdout=open(f"/tmp/vllm_{MODEL_PORT}.log", "w"), stderr=subprocess.STDOUT)

if not wait_for_server(MODEL_PORT):
    print(f"⚠️ Check log: /tmp/vllm_{MODEL_PORT}.log")

print("\n✅ Service startup sequence complete!")

In [ ]:
from pathlib import Path

log_path = Path(f"/tmp/vllm_{MODEL_PORT}.log")

with log_path.open("r", encoding="utf-8", errors="replace") as f:
    lines = f.readlines()

print("Total lines:", len(lines))
print("\n".join(lines))

In [ ]:
print("\n==================================================")
print("🌐 Create NGINX FILE")
print("==================================================")

nginx_config = """
events {
    worker_connections 1024;
}

http {
    proxy_read_timeout 300s;
    proxy_connect_timeout 300s;
    proxy_send_timeout 300s;

    server {
        listen 8080;


        # Disable buffering for SSE token streaming
        proxy_buffering off;
        proxy_http_version 1.1;
        
        proxy_set_header Connection "";
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;

        location /chat/ {
            proxy_pass http://127.0.0.1:8001/;
        }
    }
}
"""

with open("/tmp/nginx.conf", "w") as f:
    f.write(nginx_config)

if "nginx_process" not in globals():
    nginx_process = None

def stop_nginx_servers(_process, _process_name, _process_port):
    if _process_name in globals() and _process is not None:
        if _process.poll() is None:
            print(f"🛑 Stopping previous {_process_name} server (PID: {_process.pid})...")
            _process.terminate()
            try:
                _process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                _process.kill()
                _process.wait()

    result = subprocess.run(["lsof", "-t", f"-i:{_process_port}"], capture_output=True, text=True)
    if result.stdout.strip():
        pids = result.stdout.strip().splitlines()
        for pid in pids:
            subprocess.run(["kill", "-9", pid])
        time.sleep(2)
    print(f"✅ Port {_process_port} is free.")

stop_nginx_servers(nginx_process, "nginx_process", 8080)

nginx_process = subprocess.Popen([
    "nginx",
    "-c", "/tmp/nginx.conf",
    "-g", "daemon off;"
])

print("✅ NGINX running on port 8080")

In [ ]:
import re

print("\n==================================================")
print("🌐 EXPOSING PORT 8080 VIA CLOUDFLARE")
print("==================================================")

tunnel_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8080"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)



print("\nStarting tunnel...")
for line in tunnel_process.stdout:
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            print("\n🚀 SUCCESS! Your public Base URL is:")
            print(match.group(0))
            break